# Train RawNet2 from Scratch
This notebook mounts Google Drive, prepares the ASVspoof 2019 dataset on local VM storage for speed, and runs training for RawNet2.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
PROJECT_ROOT = '/content/drive/MyDrive/ANN/rawnet2-antispoofing/'
%cd {PROJECT_ROOT}
!ls

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/183UcSEiMJbMU7BYJsjiKrxfrr8t2SAV6/ANN/rawnet2-antispoofing
dataset			       model.py
data_utils_LA.py	       models
LA.zip			       __pycache__
LFCC_high_resolution_baseline  README.md
LICENSE			       requirements.txt
logs			       smoke_config_RawNet2.yaml
main.py			       SVM_fusion
model_config_RawNet2.yaml      tDCF_python


## 1. Extract Dataset to Local VM
Unzip the `LA.zip` file to `/content/dataset` for maximum performance.

In [2]:
if not os.path.exists('/content/dataset'):
    print("Extracting dataset to local VM storage (/content/dataset)...")
    !unzip -q LA.zip -d /content/dataset
    print("Extraction complete!")
else:
    print("Dataset already exists in local storage.")

Extracting dataset to local VM storage (/content/dataset)...
Extraction complete!


## 2. Install Requirements
We install compatible versions of the required libraries.

In [3]:
# Using modern compatible versions to avoid build errors
!pip install tensorboardX soundfile librosa pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.4 MB/s eta 0:00:00


## 3. Find Data Paths
Locate the `flac` files and the protocols in the local VM storage.

In [4]:
# Search for the database and protocols paths inside the extracted folder on local VM
flac_paths = glob.glob('/content/dataset/**/ASVspoof2019_LA_train/flac', recursive=True)
protocol_paths = glob.glob('/content/dataset/**/ASVspoof2019_LA_cm_protocols', recursive=True)

if flac_paths and protocol_paths:
    # database_path should be the parent of the folder containing ASVspoof2019_LA_train
    DATABASE_PATH = os.path.dirname(os.path.dirname(flac_paths[0]))
    PROTOCOLS_PATH = protocol_paths[0]
    print(f"Found DATABASE_PATH: {DATABASE_PATH}")
    print(f"Found PROTOCOLS_PATH: {PROTOCOLS_PATH}")
else:
    print("❌ Could not find expected dataset structure in /content/dataset.")

Found DATABASE_PATH: /content/dataset/LA
Found PROTOCOLS_PATH: /content/dataset/LA/ASVspoof2019_LA_cm_protocols


## 4. Smoke Training
Test the training loop with a tiny subset (10 samples) and 1 epoch.

In [ ]:
# Swap to smoke config
!cp model_config_RawNet2.yaml model_config_RawNet2.yaml.bak
!cp smoke_config_RawNet2.yaml model_config_RawNet2.yaml

# Running with --sample_size 10 so it finishes instantly for verification
!python main.py \
    --database_path {DATABASE_PATH} \
    --protocols_path {PROTOCOLS_PATH} \
    --num_epochs 1 \
    --batch_size 2 \
    --sample_size 10 \
    --comment "smoke_test"

# Restore original config
!mv model_config_RawNet2.yaml.bak model_config_RawNet2.yaml
print("Smoke Training complete!")

## 5. Real Training
Run the full training process. Automatically resumes if a checkpoint is found.

In [5]:
import glob
import os
import time
import torch

# 1. Configurations
DATABASE_PATH = "/content/dataset/LA"
PROTOCOLS_PATH = "/content/dataset/LA/ASVspoof2019_LA_cm_protocols/"
COMMENT = "full_seed" 

# 2. Find the latest valid checkpoint
def get_valid_checkpoint(comment):
    checkpoint_files = glob.glob(f"models/**/*{comment}*/*.pth", recursive=True)
    checkpoint_files.sort(key=os.path.getmtime, reverse=True)
    for ckpt in checkpoint_files:
        try:
            torch.load(ckpt, map_location='cpu', weights_only=True)
            return ckpt 
        except:
            continue
    return None

latest_ckpt = get_valid_checkpoint(COMMENT)
model_path_arg = f"--model_path {latest_ckpt}" if latest_ckpt else ""

# 3. Run Training (Set to 25 Epochs)
!python main.py \
    --database_path {DATABASE_PATH} \
    --protocols_path {PROTOCOLS_PATH} \
    --num_epochs 25 \
    --batch_size 32 \
    --comment '{COMMENT}' \
    {model_path_arg}

print("\n✅ Finished 25 epochs!")

Run seed: 609
Data seed: 609
Model loaded : models/model_logical_weighted_CCE_24_32_0.0001_full_seed/latest_mid_epoch.pth
Starting from epoch 24

Start training epoch 24
Batch [10/794] | Loss: 0.0003 | Acc: 100.00%
Batch [20/794] | Loss: 0.0002 | Acc: 100.00%
Batch [30/794] | Loss: 0.0002 | Acc: 100.00%
Batch [40/794] | Loss: 0.0002 | Acc: 100.00%
Batch [50/794] | Loss: 0.0002 | Acc: 100.00%
Batch [60/794] | Loss: 0.0005 | Acc: 99.95%
Batch [70/794] | Loss: 0.0004 | Acc: 99.96%
Batch [80/794] | Loss: 0.0004 | Acc: 99.96%
Batch [90/794] | Loss: 0.0004 | Acc: 99.97%
Batch [100/794] | Loss: 0.0004 | Acc: 99.97%
Checkpoint saved successfully
Batch [110/794] | Loss: 0.0004 | Acc: 99.97%
Batch [120/794] | Loss: 0.0003 | Acc: 99.97%
Batch [130/794] | Loss: 0.0004 | Acc: 99.98%
Batch [140/794] | Loss: 0.0006 | Acc: 99.96%
Batch [150/794] | Loss: 0.0006 | Acc: 99.96%
Batch [160/794] | Loss: 0.0005 | Acc: 99.96%
Batch [170/794] | Loss: 0.0044 | Acc: 99.91%
Batch [180/794] | Loss: 0.0079 | Acc: 9